<table>
  <tr>
    <td><div align="left"><font size="30">Modeling a perspective camera</font></div></td>
    <td><img src="https://github.com/petercorke/machinevision-toolbox-python/raw/main/docs/figs/VisionToolboxLogo_NoBackgnd@2x.png?raw=1" width="200" height="158" style="height:auto"></td>
  </tr>
</table>

(c) Peter Corke 2024

Robotics, Vision & Control: Python, see Section 13.2.2

## Configuring the Jupyter environment
We need to import some packages to help us with linear algebra (`numpy`), graphics (`matplotlib`), and machine vision (`machinevisiontoolbox`).
If you're running locally you need to have these packages installed.  If you're running on CoLab we have to first install machinevisiontoolbox which is not preinstalled, this will be a bit slow.

In [ ]:
# MVTB_BOOTSTRAP_CELL: installs the package, which takes a short time, so wait for the line "Running ..." below.  Generated by sync_bootstrap.py from _mvtb_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'machinevision-toolbox-python'
IMPORT_NAME = 'machinevisiontoolbox'
GIT_URL: str | None = 'https://github.com/petercorke/machinevision-toolbox-python'
JL_PACKAGES = ['numpy', 'scipy', 'matplotlib', 'pillow', 'typing_extensions', 'opencv-python', 'ansitable', 'mvtb-data', 'tqdm', 'requests', 'ipywidgets', 'spatialmath-python', 'pgraph-python']
JL_NO_DEPS: list[str] = []
JL_PACKAGE_NO_DEPS = True
JL_EXTRAS: dict[str, list[str]] = {}
COLAB_PACKAGES: list[str] = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE, deps=not JL_PACKAGE_NO_DEPS)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed()


In [ ]:
%matplotlib inline

import numpy as np
import math
from spatialmath import SE3
from spatialmath.base import e2h, h2e
import machinevisiontoolbox as mvtb

***

# From first principles

Let's define some parameters of our camera

In [ ]:
f = 8*1e-3     # focal length in metres
rho = 10*1e-6  # pixel side length in metres
u0 = 500       # principal point, horizontal coordinate
v0 = 500       # principal point, vertical coordinate

Next we define some matrices.  

The first is a 3x3 _intrinsic_ matrix that contains information about the camera itself:
  - focal length of the lens
  - pixel size
  - the position of the principal point in units of pixels,this is where the _principal ray_ pierces the image plane.

In [ ]:
K = np.array([  [f/rho, 0, u0], 
                [0, f/rho, v0], 
                [0, 0, 1]])
print(K)

The next is a 3x4 matrix that performs the projection, reduces the 3D world point to a 2D image plane point

In [ ]:
P0 = np.array([ [1, 0, 0, 0],
                [0, 1, 0, 0],
                [0, 0, 1, 0]])

Then finally a 4x4 _extrinsic_ matrix_ which says how the camera is positioned and oriented in space. An identity matrix has our camera at the world frame origin and looking along the world z-axis.

In [ ]:
X = np.identity(4)

We multiply these three matrices together to form the camera matrix

In [ ]:
C = K @ P0 @ np.linalg.inv(X)
C

Next we define a point in terms of its 3D coordinates (in units of metres), as a column vector

In [ ]:
P = np.array([1, -2, 5])
P

To perform the projection we need to convert our 3D coordinate to homogeneous form, which is done by the helper function `e2h`

In [ ]:
e2h(P)

Now we can premultiply this by the camera matrix to obtain the image plane coordinates in homoogeneous form

In [ ]:
C @ e2h(P)

and we use the helper function `h2e` to convert that back to Euclidean coordinates - the 2D image plane coordinates in units of pixels

In [ ]:
h2e( C @ e2h(P) )

If we move the point 0.5m in the camera's x-axis direction

In [ ]:
P1 = np.array([1.5, -2, 5])
h2e( C @ e2h(P1) )

we see that the u-coordinate in the image plane increases, as we would expect

Now we will move the camera 0.5m in the x-axis direction. We do this by setting the top right element of the _extrinsic_ matrix so as to represent a translation of 0.5m.

In [ ]:
X[0,3] = 0.5
X

Then we recompute the _camera_ matrix and project our original point onto the image plane of the moved camera.

In [ ]:
C = K @ P0 @ np.linalg.inv(X)
h2e( C @ e2h(P) )

We see that the u-coordinate has decreased as we would expect - when we move our head to the right the world appears to move to the left.

# Using the Toolbox CentralCamera class



The toolbox provide a class to make this a bit more convenient.  We have to provide the focal length `f`, pixel scale factor `rho`, the principal point `pp`, and the image dimensions `imagesize` (in this case 1000x1000)

In [ ]:
camera = mvtb.CentralCamera(f=f, rho=rho, pp=(u0, v0), imagesize=1000)
print(camera)

We can very conveniently project points to the image plane

In [ ]:
camera.project_point(P)

and we can plot it on a virtual image plane conveniently by

In [ ]:
camera.plot_point(P)

We can project the same point, but this time with the camera moved 0.5m in the x-direction

In [ ]:
camera.project_point(P, pose=SE3(0.5, 0, 0))

and we see that the u-coordinate has decreased.  If we look out along the camera's principal axis then moving the camera to the right causes the image plane point to move to the left.  Note that the vertical coordinate hasn't changed -- as expected.

_The `SE3` class constructs a matrix belonging to the group SE(3) which can be used to describe a pose in 3D space.  Such matrices are used frequently in robotics, computer vision and computer graphics._

We can extract the intrinsic matrix

In [ ]:
camera.K

and the camera matrix

In [ ]:
camera.C()

and also the camera matrix for the case where the camera is moved

In [ ]:
camera.C(pose=SE3(0.5, 0, 0))